# BM25 Sparse Index Builder - 10 Shards

Notebook này::

- BM25 chỉ index `chunk_text`
- Metadata lưu full payload để trace/citation
- Tokenizer: `underthesea.word_tokenize`
- Dùng `BM25SparseRetriever.build_from_records(...)`
- Output mỗi shard gồm `bm25_index.pkl` và `bm25_metadata.pkl`
- Mỗi lần chỉ build 1/10 corpus để tránh chạy toàn bộ 1.5M chunks một lần.

## 1. Setup

In [ ]:
!pip install -q rank_bm25 underthesea

import json
import math
import os
import pickle
import re
import shutil
import sys
import time
import unicodedata
from dataclasses import dataclass
from itertools import islice
from pathlib import Path
from typing import Any

print('Python version:', sys.version)
print('Setup complete.')

## 2. Configuration

In [ ]:
IS_KAGGLE = os.path.exists('/kaggle/input')
IS_COLAB = 'google.colab' in sys.modules

if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working')
    CHUNKS_PATH = Path('/kaggle/input/datasets/nguyenlethienlyy/text-mining-data-preprocessed/chunks-003.jsonl')
elif IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/TextMining')
    CHUNKS_PATH = PROJECT_ROOT / 'data' / 'v2' / 'chunks.jsonl'
else:
    PROJECT_ROOT = Path('.').resolve()
    CHUNKS_PATH = PROJECT_ROOT / 'data' / 'pre-processed' / 'chunks-003.jsonl'

TEXT_FIELD = 'chunk_text'
ID_FIELD = 'chunk_id'

TOTAL_CHUNKS = 1_513_376
TOTAL_SHARDS = 10
SHARD_ID = 2  # change this: 0, 1, 2, ..., 9

# Two source lines make underthesea.word_tokenize hang in shard 2.
SKIP_SOURCE_LINES = {394075, 394076} if SHARD_ID == 2 else set()

SHARD_SIZE = math.ceil(TOTAL_CHUNKS / TOTAL_SHARDS)
START_AT = SHARD_ID * SHARD_SIZE
STOP_AT = min(START_AT + SHARD_SIZE, TOTAL_CHUNKS)

OUTPUT_DIR = PROJECT_ROOT / 'bm25_sparse_shards' / f'shard_{SHARD_ID:02d}'

assert 0 <= SHARD_ID < TOTAL_SHARDS

print('Environment:', 'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local')
print('CHUNKS_PATH:', CHUNKS_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('TEXT_FIELD:', TEXT_FIELD, '| ID_FIELD:', ID_FIELD)
print(f'SHARD_ID: {SHARD_ID}/{TOTAL_SHARDS - 1}')
print(f'Range: {START_AT:,} -> {STOP_AT:,} ({STOP_AT - START_AT:,} chunks)')
print('SKIP_SOURCE_LINES:', sorted(SKIP_SOURCE_LINES))

## 3. BM25 Sparse Retriever

In [ ]:
PROVENANCE_FIELDS = (
    'chunk_id',
    'parent_unit_id',
    'id_str',
    'chunk_index_in_unit',
    'chunk_count_in_unit',
    'char_start',
    'char_end',
)


@dataclass(frozen=True)
class SearchHit:
    point_id: str
    score: float
    payload: dict[str, Any]

    def provenance(self) -> dict[str, Any]:
        return {k: self.payload.get(k) for k in PROVENANCE_FIELDS if k in self.payload}


def simple_tokenize(text: str) -> list[str]:
    text = unicodedata.normalize('NFC', text).lower()
    text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
    return [tok for tok in text.split() if len(tok) > 1]


def get_tokenizer():
    try:
        from underthesea import word_tokenize

        def _tokenize(text: str) -> list[str]:
            segmented = word_tokenize(text, format='text')
            return simple_tokenize(segmented)

        print('Using underthesea word_tokenize for BM25 tokenization')
        return _tokenize
    except ImportError:
        print('underthesea not available; using simple whitespace tokenizer')
        return simple_tokenize


class BM25SparseRetriever:
    def __init__(self, *, bm25, chunk_ids, payloads, tokenizer, text_field='chunk_text'):
        self._bm25 = bm25
        self._chunk_ids = chunk_ids
        self._payloads = payloads
        self._tokenizer = tokenizer
        self._text_field = text_field

    @property
    def total_documents(self) -> int:
        return len(self._chunk_ids)

    @classmethod
    def build_from_records(cls, records, *, text_field='chunk_text', id_field='chunk_id', tokenizer=None):
        from rank_bm25 import BM25Okapi

        if tokenizer is None:
            tokenizer = get_tokenizer()

        chunk_ids, payloads, tokenized = [], [], []
        empty_text = 0

        from tqdm.auto import tqdm

        for i, record in enumerate(tqdm(records, desc='Tokenizing'), start=1):
            
            if (i%1000==0):
                print(f'tokenized {i:,}/{len(records):,}')
            chunk_ids.append(str(record.get(id_field) or ''))
            payloads.append(record)

            text = str(record.get(text_field) or '')
            if not text.strip():
                empty_text += 1
            tokenized.append(tokenizer(text))

        if empty_text:
            print(f'Warning: {empty_text} records had empty {text_field!r}')

        if tokenized:
            sample_n = min(12, len(tokenized[0]))
            print(
                f'BM25 corpus field={text_field!r} only | '
                f'sample_0 tokens={len(tokenized[0])} first={tokenized[0][:sample_n]}'
            )
            print(f'Payloads store full records; provenance keys: {list(PROVENANCE_FIELDS)}')

        bm25 = BM25Okapi(tokenized)
        return cls(
            bm25=bm25,
            chunk_ids=chunk_ids,
            payloads=payloads,
            tokenizer=tokenizer,
            text_field=text_field,
        )

    def search(self, query: str, *, top_k: int = 20) -> list[SearchHit]:
        tokenized_query = self._tokenizer(query)
        scores = self._bm25.get_scores(tokenized_query)
        top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

        hits = []
        for idx in top_indices:
            score = float(scores[idx])
            if score <= 0:
                continue
            hits.append(SearchHit(
                point_id=self._chunk_ids[idx],
                score=score,
                payload=self._payloads[idx],
            ))
        return hits

    def search_with_latency(self, query: str, *, top_k: int = 20):
        t0 = time.perf_counter()
        hits = self.search(query, top_k=top_k)
        return hits, time.perf_counter() - t0

    def save(self, index_dir: Path):
        index_dir = Path(index_dir)
        index_dir.mkdir(parents=True, exist_ok=True)

        index_path = index_dir / 'bm25_index.pkl'
        meta_path = index_dir / 'bm25_metadata.pkl'

        with index_path.open('wb') as f:
            pickle.dump(self._bm25, f, protocol=pickle.HIGHEST_PROTOCOL)

        with meta_path.open('wb') as f:
            pickle.dump(
                {
                    'chunk_ids': self._chunk_ids,
                    'payloads': self._payloads,
                    'text_field': self._text_field,
                    'provenance_fields': list(PROVENANCE_FIELDS),
                },
                f,
                protocol=pickle.HIGHEST_PROTOCOL,
            )

        print(f'Saved BM25 index to {index_path} ({len(self._chunk_ids):,} docs)')
        print(f'Saved BM25 metadata to {meta_path}')
        return index_path, meta_path

    @classmethod
    def load(cls, index_dir: Path, *, tokenizer=None):
        index_dir = Path(index_dir)
        if tokenizer is None:
            tokenizer = get_tokenizer()

        with (index_dir / 'bm25_index.pkl').open('rb') as f:
            bm25 = pickle.load(f)
        with (index_dir / 'bm25_metadata.pkl').open('rb') as f:
            meta = pickle.load(f)

        return cls(
            bm25=bm25,
            chunk_ids=meta['chunk_ids'],
            payloads=meta['payloads'],
            tokenizer=tokenizer,
            text_field=meta.get('text_field', 'chunk_text'),
        )


print('BM25SparseRetriever defined.')

## 4. Load One Shard

In [ ]:
print(f'Loading shard {SHARD_ID}: rows {START_AT:,} -> {STOP_AT:,}')
if not CHUNKS_PATH.exists():
    raise FileNotFoundError(f'Chunks file not found: {CHUNKS_PATH}')

chunks = []
skipped_rows = []
t0 = time.perf_counter()

with CHUNKS_PATH.open('r', encoding='utf-8') as f:
    shard_lines = islice(f, START_AT, STOP_AT)
    for source_line_no, line in enumerate(shard_lines, start=START_AT + 1):
        line = line.strip()
        if not line:
            continue

        record = json.loads(line)
        if source_line_no in SKIP_SOURCE_LINES:
            skipped_rows.append({
                'source_line_no': source_line_no,
                'chunk_id': record.get(ID_FIELD),
            })
            continue

        chunks.append(record)

load_time = time.perf_counter() - t0
print(f'Loaded {len(chunks):,} chunks in {load_time:.2f}s')
print('Skipped rows:', skipped_rows)

if chunks:
    sample = chunks[0]
    print(f'Sample fields: {list(sample.keys())[:20]}')
    print(f'Sample chunk_id: {sample.get(ID_FIELD)}')
    print(f'Sample {TEXT_FIELD}[:200]: {str(sample.get(TEXT_FIELD) or "")[:200]}')

## 5. Build BM25

In [ ]:
print(f'Building BM25 on field {TEXT_FIELD!r} ({len(chunks):,} records) ...')
t0 = time.perf_counter()

sparse_retriever = BM25SparseRetriever.build_from_records(
    chunks,
    text_field=TEXT_FIELD,
    id_field=ID_FIELD,
)

build_time = time.perf_counter() - t0
print(f'BM25 shard built: {sparse_retriever.total_documents:,} documents in {build_time:.2f}s')

## 6. Save

In [ ]:
index_path, meta_path = sparse_retriever.save(OUTPUT_DIR)

manifest = {
    'shard_id': SHARD_ID,
    'total_shards': TOTAL_SHARDS,
    'start_at': START_AT,
    'stop_at': STOP_AT,
    'documents': sparse_retriever.total_documents,
    'skipped_rows': skipped_rows,
    'chunks_path': str(CHUNKS_PATH),
    'text_field': TEXT_FIELD,
    'id_field': ID_FIELD,
    'build_time_sec': round(build_time, 2),
    'load_time_sec': round(load_time, 2),
}

(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

print('\nOutput files:')
for p in (index_path, meta_path, OUTPUT_DIR / 'manifest.json'):
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f'  {p.name}: {size_mb:.2f} MB - {p}')

## 7. Smoke Test

In [ ]:
print('Reloading from disk ...')
loaded = BM25SparseRetriever.load(OUTPUT_DIR)
print(f'Loaded {loaded.total_documents:,} documents')
print(f'Index text field: {loaded._text_field!r}')

test_q = 'Điều kiện để người lao động đơn phương chấm dứt hợp đồng lao động'
test_hits, test_lat = loaded.search_with_latency(test_q, top_k=5)

print(f'\nQuery: {test_q}')
print(f'Latency: {test_lat:.4f}s')
print(f'Results: {len(test_hits)}')

for rank, hit in enumerate(test_hits, start=1):
    prov = hit.provenance()
    cid = prov.get('chunk_id') or hit.payload.get(ID_FIELD, hit.point_id)
    parent = prov.get('parent_unit_id')
    id_str = prov.get('id_str')
    preview = str(hit.payload.get(TEXT_FIELD) or '')[:120].replace('\n', ' ')

    print(f'  [{rank}] score={hit.score:.4f} chunk_id={cid}')
    print(f'       lineage: parent_unit_id={parent} id_str={id_str}')
    print(f'       text: {preview}...')

## 8. Zip This Shard

In [ ]:
zip_path = shutil.make_archive(str(OUTPUT_DIR), 'zip', OUTPUT_DIR)
print('saved:', zip_path)